# SFT + DPO Training & Assessment Pipeline

Full pipeline:
1. **SFT Training** (Phase 1: CPT + Phase 2: SFT)
2. **Full Test Set Assessment** with detailed report
3. **Generate DPO Preference Pairs** (auto-ranked by metrics)
4. **DPO Training** on top of SFT adapter
5. **Post-DPO Assessment** and compare SFT vs SFT+DPO
6. **Save everything** to Google Drive

**Requirements:** GPU runtime (T4 or better). Runtime > Change runtime type > GPU.

## 1. Setup & Install

In [ ]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes trl sentence-transformers matplotlib pandas anthropic codebleu --no-build-isolation
!pip install -q -U flash-attn --no-build-isolation 2>/dev/null || echo 'flash-attn not available, will use eager attention'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# === Google Drive — final adapters + reports only (no checkpoints) ===
DRIVE_ROOT = '/content/drive/MyDrive/cocos2dx-finetune'
DRIVE_SFT_ADAPTER = f'{DRIVE_ROOT}/sft-adapter'
DRIVE_DPO_ADAPTER = f'{DRIVE_ROOT}/dpo-adapter'
DRIVE_REPORTS = f'{DRIVE_ROOT}/reports'
DRIVE_DPO_DATA = f'{DRIVE_ROOT}/dpo-preference-data'

for d in [DRIVE_SFT_ADAPTER, DRIVE_DPO_ADAPTER, DRIVE_REPORTS, DRIVE_DPO_DATA]:
    os.makedirs(d, exist_ok=True)

# === Colab local — checkpoints stay here, lost on session end (that's fine) ===
LOCAL_CPT_DIR = '/content/checkpoints/phase1-cpt'
LOCAL_SFT_DIR = '/content/checkpoints/phase2-sft'
LOCAL_DPO_DIR = '/content/checkpoints/phase3-dpo'
for d in [LOCAL_CPT_DIR, LOCAL_SFT_DIR, LOCAL_DPO_DIR]:
    os.makedirs(d, exist_ok=True)

# Empty Drive trash to free space
try:
    !find /content/drive/.Trash* -mindepth 1 -delete 2>/dev/null
    print('Drive trash emptied.')
except:
    pass

print(f'Drive: adapters + reports at {DRIVE_ROOT}')
print(f'Local: checkpoints at /content/checkpoints (not saved to Drive)')

In [ ]:
import os

REPO = 'nmnhut-it/fine-tune-cocoos'
if not os.path.exists('fine-tune-cocoos'):
    !git clone https://github.com/{REPO}.git

os.chdir('fine-tune-cocoos')

## 2. Load & Clean Data

In [ ]:
from datasets import load_dataset, Dataset
import json, re, os

# RAFT training data: each example has instruction + context (retrieved doc) + output
train_ds = load_dataset('json', data_files='data/train-raft.jsonl', split='train')
test_ds  = load_dataset('json', data_files='data/test.jsonl', split='train')

print(f'Train (RAFT): {len(train_ds)} examples')
print(f'Test:         {len(test_ds)} examples')
print(f'Train columns: {train_ds.column_names}')
print(f'Sample context length: {len(train_ds[0].get("context",""))} chars')

In [ ]:
BROKEN_PREFIXES = [
    r"^Is it possible to how ",
    r"^I've been trying to how ",
    r"^So I need to what's ",
    r"^What do I need to know about What ",
    r"^What's the proper way to handle What ",
    r"^How exactly do you how ",
    r"^What's a clean way to how ",
]

def clean_instruction(text):
    for pattern in BROKEN_PREFIXES:
        if re.match(pattern, text, re.IGNORECASE):
            text = re.sub(pattern, '', text, flags=re.IGNORECASE)
            text = text[0].upper() + text[1:] if text else text
    return text.strip()

def dedup_dataset(ds):
    seen, keep = set(), []
    for i, ex in enumerate(ds):
        key = ex['output'].strip().lower()[:200]
        if key not in seen:
            seen.add(key)
            keep.append(i)
    return ds.select(keep)

train_ds = train_ds.map(lambda x: {**x, 'instruction': clean_instruction(x['instruction'])})
before = len(train_ds)
train_ds = dedup_dataset(train_ds)
print(f'Train after dedup: {before} -> {len(train_ds)}')
print(f'Test: {len(test_ds)} examples')
full_train = train_ds  # alias used in later cells

## 3. Model & Tokenizer Setup

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = 'Qwen/Qwen3-8B-Base'

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None or tokenizer.pad_token == tokenizer.eos_token:
    tokenizer.add_special_tokens({'pad_token': '<|pad|>'})
tokenizer.padding_side = 'right'

attn_impl = 'flash_attention_2'
try:
    import flash_attn
except ImportError:
    attn_impl = 'eager'
    print('flash-attn not installed, using eager attention')

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=compute_dtype,
    attn_implementation=attn_impl,
)
model.resize_token_embeddings(len(tokenizer))
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    use_dora=True,
)

model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()
model.print_trainable_parameters()

## 4. Tokenization (Two-Phase)

In [ ]:
RAFT_TEMPLATE = """Below is an instruction that describes a task. Use the provided documentation context to write an accurate response.

### Documentation Context:
{context}

### Instruction:
{instruction}

### Response:
{output}"""

RAFT_PROMPT = """Below is an instruction that describes a task. Use the provided documentation context to write an accurate response.

### Documentation Context:
{context}

### Instruction:
{instruction}

### Response:
"""

# Fallback for test set (no context field)
ALPACA_TEMPLATE = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
{output}"""

ALPACA_PROMPT = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
"""

MAX_LENGTH = 1536  # longer to fit context + instruction + response
eos_id = tokenizer.eos_token_id


def tokenize_full_text(example):
    """Phase 1 (CPT): full-text loss on context + instruction + response."""
    context = example.get('context', '')
    if context:
        full = RAFT_TEMPLATE.format(context=context, instruction=example['instruction'],
                                    output=example['output'])
    else:
        full = ALPACA_TEMPLATE.format(instruction=example['instruction'],
                                      output=example['output'])
    tokens = tokenizer(full, truncation=True, max_length=MAX_LENGTH - 1, padding=False)
    tokens['input_ids'] = tokens['input_ids'] + [eos_id]
    tokens['attention_mask'] = tokens['attention_mask'] + [1]
    tokens['labels'] = list(tokens['input_ids'])
    return tokens


def tokenize_masked(example):
    """Phase 2 (SFT): loss only on response tokens, context+instruction masked."""
    context = example.get('context', '')
    if context:
        full = RAFT_TEMPLATE.format(context=context, instruction=example['instruction'],
                                    output=example['output'])
        prompt = RAFT_PROMPT.format(context=context, instruction=example['instruction'])
    else:
        full = ALPACA_TEMPLATE.format(instruction=example['instruction'],
                                      output=example['output'])
        prompt = ALPACA_PROMPT.format(instruction=example['instruction'])

    tokens = tokenizer(full, truncation=True, max_length=MAX_LENGTH - 1, padding=False)
    tokens['input_ids'] = tokens['input_ids'] + [eos_id]
    tokens['attention_mask'] = tokens['attention_mask'] + [1]
    prompt_tokens = tokenizer(prompt, truncation=True, max_length=MAX_LENGTH)
    prompt_len = len(prompt_tokens['input_ids'])
    tokens['labels'] = [-100] * prompt_len + tokens['input_ids'][prompt_len:]
    return tokens


# Test set has no context field — use alpaca format for evaluation consistency
def tokenize_test(example):
    full = ALPACA_TEMPLATE.format(instruction=example['instruction'],
                                  output=example.get('output', ''))
    tokens = tokenizer(full, truncation=True, max_length=MAX_LENGTH - 1, padding=False)
    tokens['input_ids'] = tokens['input_ids'] + [eos_id]
    tokens['attention_mask'] = tokens['attention_mask'] + [1]
    tokens['labels'] = list(tokens['input_ids'])
    return tokens


remove_cols = [c for c in full_train.column_names]
remove_test_cols = [c for c in test_ds.column_names]

cpt_train = full_train.map(tokenize_full_text, remove_columns=remove_cols)
cpt_test  = test_ds.map(tokenize_test, remove_columns=remove_test_cols)
sft_train = full_train.map(tokenize_masked, remove_columns=remove_cols)
sft_test  = test_ds.map(tokenize_test, remove_columns=remove_test_cols)

print(f'CPT data: {len(cpt_train)} train, {len(cpt_test)} test')
print(f'SFT data: {len(sft_train)} train, {len(sft_test)} test')
print(f'Avg CPT seq len: {sum(len(x["input_ids"]) for x in cpt_train) / len(cpt_train):.0f} tokens')

## 5. SFT Training (Phase 1: CPT + Phase 2: SFT)

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Any
from transformers import TrainingArguments, Trainer
import glob

@dataclass
class LabelPreservingCollator:
    tokenizer: Any
    max_length: int = 2048

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        pad_id = self.tokenizer.pad_token_id
        batch = {}
        for key in ('input_ids', 'attention_mask', 'labels'):
            seqs = [f[key] for f in features]
            pad_val = -100 if key == 'labels' else (0 if key == 'attention_mask' else pad_id)
            max_len = min(max(len(s) for s in seqs), self.max_length)
            padded = []
            for s in seqs:
                diff = max_len - len(s)
                padded.append(s + [pad_val] * diff if diff > 0 else s[:max_len])
            batch[key] = torch.tensor(padded, dtype=torch.long)
        return batch

use_bf16 = torch.cuda.is_bf16_supported()

def make_trainer(train_data, test_data, output_dir, num_epochs, lr):
    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=1,   # reduced from 2 to avoid OOM on T4
        gradient_accumulation_steps=16,  # increased to keep effective batch=16
        learning_rate=lr,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type='cosine',
        bf16=use_bf16,
        fp16=not use_bf16,
        gradient_checkpointing=True,
        logging_steps=10,
        save_strategy='steps',
        save_steps=100,
        save_total_limit=3,
        report_to='none',
        dataloader_pin_memory=False,
        neftune_noise_alpha=5.0,
    )
    return Trainer(
        model=model, args=args,
        train_dataset=train_data, eval_dataset=test_data,
        data_collator=LabelPreservingCollator(tokenizer),
    )

In [ ]:
# === Phase 1: Continued Pre-Training (full-text loss on RAFT context+instruction+response) ===
print('=' * 60)
print('PHASE 1: Continued Pre-Training (RAFT)')
print('=' * 60)

trainer = make_trainer(cpt_train, cpt_test, LOCAL_CPT_DIR, num_epochs=5, lr=2e-4)
checkpoints = sorted(glob.glob(f'{LOCAL_CPT_DIR}/checkpoint-*'))
resume = checkpoints[-1] if checkpoints else None
if resume:
    print(f'Resuming from {resume}')
trainer.train(resume_from_checkpoint=resume)

In [ ]:
# === Phase 2: Supervised Fine-Tuning (response-only loss) ===
print('=' * 60)
print('PHASE 2: Supervised Fine-Tuning (RAFT, response-only loss)')
print('=' * 60)

trainer = make_trainer(sft_train, sft_test, LOCAL_SFT_DIR, num_epochs=3, lr=5e-5)
checkpoints = sorted(glob.glob(f'{LOCAL_SFT_DIR}/checkpoint-*'))
resume = checkpoints[-1] if checkpoints else None
if resume:
    print(f'Resuming from {resume}')
trainer.train(resume_from_checkpoint=resume)

In [ ]:
# Save SFT adapter
model.save_pretrained(DRIVE_SFT_ADAPTER)
tokenizer.save_pretrained(DRIVE_SFT_ADAPTER)
print(f'SFT adapter saved to {DRIVE_SFT_ADAPTER}')

---
## 6. Post-SFT Assessment (Full Test Set)

Generate on every test example, compute metrics, save detailed report.

In [ ]:
import json, re, os
from tqdm import tqdm

# === New metrics (replaces BLEU/ROUGE) ===
# Load symbol whitelist for hallucination detection
_whitelist = None
def load_whitelist():
    global _whitelist
    if _whitelist is None:
        _whitelist = set(json.load(open('data/api_symbols.json'))['symbols'])
    return _whitelist

SYM_PAT = re.compile(r'\b((?:cc|ccui|ccs|sp)\.[A-Za-z][A-Za-z0-9_.]*)')

def check_symbols(prediction, reference):
    whitelist = load_whitelist()
    pred_syms = set(SYM_PAT.findall(prediction))
    ref_syms  = set(SYM_PAT.findall(reference))
    hallucinated = pred_syms - whitelist
    valid_pred   = pred_syms & whitelist
    precision = len(valid_pred) / len(pred_syms) if pred_syms else 1.0
    recall    = len(ref_syms & pred_syms) / len(ref_syms) if ref_syms else None
    f1 = (2 * precision * recall / (precision + recall)
          if recall is not None and (precision + recall) > 0 else None)
    return {
        'hallucinated_symbols': sorted(hallucinated),
        'symbol_precision': round(precision, 4),
        'symbol_recall': round(recall, 4) if recall is not None else None,
        'symbol_f1': round(f1, 4) if f1 is not None else None,
        'has_hallucination': len(hallucinated) > 0,
    }

def compute_codebleu(prediction, reference):
    try:
        from codebleu import calc_codebleu
        r = calc_codebleu([reference], [prediction], lang='javascript',
                          weights=(0.25, 0.25, 0.25, 0.25))
        return round(r['codebleu'], 4)
    except Exception:
        return None

def compute_metrics(prediction, reference):
    sym = check_symbols(prediction, reference)
    cb  = compute_codebleu(prediction, reference)
    return {**sym, 'codebleu': cb}

def generate_response(mdl, instruction, max_new_tokens=512):
    prompt = ALPACA_PROMPT.format(instruction=instruction)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True,
                       max_length=2048).to(mdl.device)
    with torch.no_grad():
        output = mdl.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.7, top_p=0.9, do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:],
                            skip_special_tokens=True)

def run_full_test(mdl, test_dataset, label, save_path):
    results = []
    if os.path.exists(save_path):
        with open(save_path) as f:
            results = json.load(f)
        print(f'Resuming {label} from {len(results)} completed examples')
    completed = {r['index'] for r in results}

    for i in tqdm(range(len(test_dataset)), desc=f'Testing ({label})'):
        if i in completed:
            continue
        ex = test_dataset[i]
        prediction = generate_response(mdl, ex['instruction'])
        metrics = compute_metrics(prediction, ex['output'])
        results.append({
            'index': i,
            'instruction': ex['instruction'],
            'reference': ex['output'],
            'prediction': prediction,
            **metrics,
        })
        if len(results) % 10 == 0:
            with open(save_path, 'w') as f:
                json.dump(results, f)

    with open(save_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'{label}: {len(results)} examples tested, saved to {save_path}')
    return results

print('Test functions ready.')

In [ ]:
# Run SFT test
model.eval()
sft_results = run_full_test(
    model, test_ds, 'SFT',
    f'{DRIVE_REPORTS}/sft_test_results.json'
)

In [ ]:
def avg(values):
    valid = [v for v in values if v is not None]
    return sum(valid) / len(valid) if valid else 0

def print_summary(results, label):
    has_cb = any(r.get('codebleu') is not None for r in results)
    print(f'\n{"=" * 55}\n  {label} -- {len(results)} examples\n{"=" * 55}')
    print(f'  Symbol Precision:   {avg([r["symbol_precision"] for r in results]):.4f}')
    print(f'  Symbol Recall:      {avg([r["symbol_recall"] for r in results]):.4f}')
    print(f'  Symbol F1:          {avg([r["symbol_f1"] for r in results]):.4f}')
    print(f'  Hallucination %:    {avg([1 if r["has_hallucination"] else 0 for r in results])*100:.1f}%')
    if has_cb:
        print(f'  CodeBLEU:           {avg([r["codebleu"] for r in results]):.4f}')

print_summary(sft_results, 'POST-SFT')

---
## 7. Generate DPO Preference Pairs

For each training prompt, generate N completions, score against reference,
pick best as `chosen` and worst as `rejected`.

In [ ]:
import random
random.seed(42)

NUM_SAMPLES_PER_PROMPT = 5
DPO_SUBSET_SIZE = 1500
DPO_DATA_PATH = f'{DRIVE_DPO_DATA}/dpo_pairs.json'

dpo_prompt_indices = random.sample(range(len(full_train)), min(DPO_SUBSET_SIZE, len(full_train)))
print(f'Will generate DPO pairs for {len(dpo_prompt_indices)} prompts, {NUM_SAMPLES_PER_PROMPT} samples each')

In [ ]:
def score_completion(prediction, reference):
    """Combined score for ranking completions."""
    m = compute_metrics(prediction, reference)
    api_score = m['api_id_f1'] if m['api_id_f1'] is not None else 0.5
    return 0.4 * m['bleu'] + 0.3 * m['rouge_l'] + 0.3 * api_score

def generate_n_samples(mdl, instruction, n=5, max_new_tokens=512):
    """Generate N diverse completions with higher temperature."""
    prompt = ALPACA_PROMPT.format(instruction=instruction)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(mdl.device)
    samples = []
    for _ in range(n):
        with torch.no_grad():
            output = mdl.generate(
                **inputs, max_new_tokens=max_new_tokens,
                temperature=0.9, top_p=0.95, do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
            )
        text = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        samples.append(text)
    return samples

# Resume support
dpo_pairs = []
if os.path.exists(DPO_DATA_PATH):
    with open(DPO_DATA_PATH) as f:
        dpo_pairs = json.load(f)
    print(f'Resuming DPO generation from {len(dpo_pairs)} pairs')
completed_dpo = {p['prompt_index'] for p in dpo_pairs}

model.eval()
for idx in tqdm(dpo_prompt_indices, desc='Generating DPO pairs'):
    if idx in completed_dpo:
        continue
    ex = full_train[idx]
    instruction = ex['instruction']
    reference = ex['output']

    samples = generate_n_samples(model, instruction, n=NUM_SAMPLES_PER_PROMPT)
    scored = [(s, score_completion(s, reference)) for s in samples]
    scored.sort(key=lambda x: x[1], reverse=True)

    best = scored[0]
    worst = scored[-1]

    # Only keep pairs with meaningful score difference
    if best[1] - worst[1] > 0.05:
        prompt_text = ALPACA_PROMPT.format(instruction=instruction)
        dpo_pairs.append({
            'prompt_index': idx,
            'prompt': prompt_text,
            'chosen': best[0],
            'rejected': worst[0],
            'chosen_score': best[1],
            'rejected_score': worst[1],
        })

    if len(dpo_pairs) % 50 == 0:
        with open(DPO_DATA_PATH, 'w') as f:
            json.dump(dpo_pairs, f)

with open(DPO_DATA_PATH, 'w') as f:
    json.dump(dpo_pairs, f, indent=2)

print(f'\nGenerated {len(dpo_pairs)} DPO preference pairs')
print(f'Avg chosen score: {avg([p["chosen_score"] for p in dpo_pairs]):.4f}')
print(f'Avg rejected score: {avg([p["rejected_score"] for p in dpo_pairs]):.4f}')
print(f'Avg score gap: {avg([p["chosen_score"] - p["rejected_score"] for p in dpo_pairs]):.4f}')

---
## 8. DPO Training

In [ ]:
from trl import DPOTrainer, DPOConfig
from datasets import Dataset

dpo_dataset = Dataset.from_dict({
    'prompt': [p['prompt'] for p in dpo_pairs],
    'chosen': [p['chosen'] for p in dpo_pairs],
    'rejected': [p['rejected'] for p in dpo_pairs],
})

dpo_split = dpo_dataset.train_test_split(test_size=0.1, seed=42)
print(f'DPO train: {len(dpo_split["train"])}, held-out: {len(dpo_split["test"])}')

In [ ]:
dpo_config = DPOConfig(
    output_dir=LOCAL_DPO_DIR,  # local Colab, not Drive
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-6,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    bf16=use_bf16,
    fp16=not use_bf16,
    gradient_checkpointing=True,
    logging_steps=10,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    report_to='none',
    max_length=1024,
    max_prompt_length=512,
    beta=0.1,
    loss_type='sigmoid',
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_split['train'],
    eval_dataset=dpo_split['test'],
    processing_class=tokenizer,
)

checkpoints = sorted(glob.glob(f'{LOCAL_DPO_DIR}/checkpoint-*'))
resume = checkpoints[-1] if checkpoints else None
if resume:
    print(f'Resuming DPO from {resume}')

print('Starting DPO training...')
dpo_trainer.train(resume_from_checkpoint=resume)

In [ ]:
model.save_pretrained(DRIVE_DPO_ADAPTER)
tokenizer.save_pretrained(DRIVE_DPO_ADAPTER)
print(f'DPO adapter saved to {DRIVE_DPO_ADAPTER}')

---
## 9. Post-DPO Assessment (Full Test Set)

In [ ]:
model.eval()
dpo_results = run_full_test(
    model, test_ds, 'DPO',
    f'{DRIVE_REPORTS}/dpo_test_results.json'
)

---
## 10. Comparison Report: SFT vs SFT+DPO

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Reload results if needed
if not sft_results:
    with open(f'{DRIVE_REPORTS}/sft_test_results.json') as f:
        sft_results = json.load(f)
if not dpo_results:
    with open(f'{DRIVE_REPORTS}/dpo_test_results.json') as f:
        dpo_results = json.load(f)

def summarize_metrics(results):
    return {
        'Symbol Precision': avg([r['symbol_precision'] for r in results]),
        'Symbol Recall':    avg([r['symbol_recall'] for r in results]),
        'Symbol F1':        avg([r['symbol_f1'] for r in results]),
        'Hallucination-free %': 1 - avg([1 if r['has_hallucination'] else 0 for r in results]),
        'CodeBLEU':         avg([r.get('codebleu') for r in results]),
    }

sft_summary = summarize_metrics(sft_results)
dpo_summary = summarize_metrics(dpo_results)

print(f'{"Metric":<25} {"SFT":>10} {"SFT+DPO":>10} {"Delta":>10}')
print('-' * 57)
for key in sft_summary:
    s, d = sft_summary[key], dpo_summary[key]
    delta = d - s
    arrow = '+' if delta > 0 else ''
    print(f'{key:<25} {s:>10.4f} {d:>10.4f} {arrow}{delta:>9.4f}')

In [ ]:
sft_by_idx = {r['index']: r for r in sft_results}
dpo_by_idx = {r['index']: r for r in dpo_results}

rows = []
for idx in sorted(sft_by_idx.keys()):
    s = sft_by_idx[idx]
    d = dpo_by_idx.get(idx)
    if d is None:
        continue
    sft_s = s.get('symbol_f1') or 0
    dpo_s = d.get('symbol_f1') or 0
    rows.append({
        'idx': idx,
        'instruction': s['instruction'][:80] + '...',
        'sft_sym_f1':  sft_s,
        'dpo_sym_f1':  dpo_s,
        'sft_codebleu': s.get('codebleu'),
        'dpo_codebleu': d.get('codebleu'),
        'sft_hallucinated': s.get('has_hallucination', False),
        'dpo_hallucinated': d.get('has_hallucination', False),
        'winner': 'DPO' if dpo_s > sft_s else 'SFT',
    })

df = pd.DataFrame(rows)
dpo_wins = (df['winner'] == 'DPO').mean() * 100
sft_wins = (df['winner'] == 'SFT').mean() * 100
print(f'\nWin rate -- DPO: {dpo_wins:.1f}%  SFT: {sft_wins:.1f}%')
df.to_csv(f'{DRIVE_REPORTS}/per_example_comparison.csv', index=False)
print(f'Saved to {DRIVE_REPORTS}/per_example_comparison.csv')
df.head(20)

In [ ]:
metrics_names = list(sft_summary.keys())
sft_vals = [sft_summary[k] for k in metrics_names]
dpo_vals = [dpo_summary[k] for k in metrics_names]

x = range(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar([i - width/2 for i in x], sft_vals, width, label='SFT', color='#4CAF50')
bars2 = ax.bar([i + width/2 for i in x], dpo_vals, width, label='SFT + DPO', color='#FF9800')

ax.set_ylabel('Score')
ax.set_title('SFT vs SFT + DPO — Full Test Set (RAFT Training)')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names, rotation=15, ha='right')
ax.legend()
ax.set_ylim(0, 1)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h),
               xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
chart_path = f'{DRIVE_REPORTS}/sft_vs_dpo_chart.png'
plt.savefig(chart_path, dpi=150)
plt.show()
print(f'Chart saved to {chart_path}')

In [ ]:
# === Side-by-Side Examples ===
from IPython.display import display, HTML
import html

def show_sft_vs_dpo(idx):
    s = sft_by_idx[idx]
    d = dpo_by_idx[idx]
    display(HTML(f"""
    <div style='border:1px solid #ccc; padding:12px; margin:8px 0; border-radius:8px'>
    <h3>Test Example {idx}</h3>
    <p><b>Instruction:</b> {html.escape(s['instruction'][:300])}</p>
    <table style='width:100%; border-collapse:collapse'>
    <tr>
      <th style='width:50%; border:1px solid #ddd; padding:8px; background:#e8f5e9'>SFT (BLEU={s['bleu']:.3f}, ROUGE={s['rouge_l']:.3f})</th>
      <th style='width:50%; border:1px solid #ddd; padding:8px; background:#fff3e0'>SFT+DPO (BLEU={d['bleu']:.3f}, ROUGE={d['rouge_l']:.3f})</th>
    </tr>
    <tr>
      <td style='border:1px solid #ddd; padding:8px; vertical-align:top'><pre style='white-space:pre-wrap; font-size:11px'>{html.escape(s['prediction'][:800])}</pre></td>
      <td style='border:1px solid #ddd; padding:8px; vertical-align:top'><pre style='white-space:pre-wrap; font-size:11px'>{html.escape(d['prediction'][:800])}</pre></td>
    </tr>
    <tr><td colspan='2' style='border:1px solid #ddd; padding:8px; background:#e3f2fd'>
      <b>Reference:</b><pre style='white-space:pre-wrap; font-size:11px'>{html.escape(s['reference'][:500])}</pre>
    </td></tr>
    </table></div>
    """))

df_sorted = df.copy()
df_sorted['rouge_delta'] = df_sorted['dpo_rouge'] - df_sorted['sft_rouge']

print('=== Top 5 DPO Improvements ===')
for idx in df_sorted.nlargest(5, 'rouge_delta')['idx'].tolist():
    show_sft_vs_dpo(idx)

print('\n=== Top 5 DPO Regressions ===')
for idx in df_sorted.nsmallest(5, 'rouge_delta')['idx'].tolist():
    show_sft_vs_dpo(idx)

---
## 11. Save Full Report

In [ ]:
report = {
    'model': MODEL_ID,
    'training': 'RAFT (Retrieval Augmented Fine-Tuning)',
    'num_test_examples': len(test_ds),
    'num_train_examples': len(full_train),
    'num_dpo_pairs': len(dpo_pairs),
    'sft_metrics': sft_summary,
    'dpo_metrics': dpo_summary,
    'delta': {k: dpo_summary[k] - sft_summary[k] for k in sft_summary},
    'dpo_win_rate': dpo_wins / 100,
    'sft_win_rate': sft_wins / 100,
    'training_config': {
        'raft_oracle_ratio': 0.7,
        'sft_phase1_epochs': 5,
        'sft_phase1_lr': 2e-4,
        'sft_phase2_epochs': 3,
        'sft_phase2_lr': 5e-5,
        'dpo_epochs': 2,
        'dpo_lr': 5e-6,
        'dpo_beta': 0.1,
        'lora_r': 64,
        'lora_alpha': 128,
        'max_length': MAX_LENGTH,
    },
}

report_path = f'{DRIVE_REPORTS}/full_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print(f'\nFull report saved to {report_path}')
print(f'\nDrive storage used:')
for fname in sorted(os.listdir(DRIVE_REPORTS)):
    size = os.path.getsize(f'{DRIVE_REPORTS}/{fname}') / 1024
    print(f'  {fname} ({size:.1f} KB)')

---
## 12. (Optional) Push to Hugging Face Hub

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()
# model.push_to_hub('your-username/cocos2dx-coder-dpo')
# tokenizer.push_to_hub('your-username/cocos2dx-coder-dpo')